# Embedding & Visualization

This notebook demonstrates how to use associo's `embedding` function to project items into 2D space using t-SNE, then visualize them with matplotlib.

Pipeline:
1. Compute associations → get Jaccard similarity
2. Convert similarity to distance
3. Run t-SNE embedding
4. Visualize items colored by community

In [ ]:
import polars as pl
import random
from associo import combinatorial_associations, communities, embedding

## 1. Prepare Data

In [ ]:
random.seed(42)

product_groups = {
    "breakfast": ["cereal", "milk", "yogurt", "granola", "orange_juice", "toast", "jam"],
    "italian": ["pasta", "tomato_sauce", "parmesan", "olive_oil", "garlic", "basil", "mozzarella"],
    "snacks": ["chips", "salsa", "beer", "pretzels", "popcorn", "soda", "nuts"],
    "baking": ["flour", "sugar", "eggs", "butter", "vanilla", "baking_powder", "chocolate"],
}

orders = []
for order_id in range(1, 1001):
    items = set()
    for group in random.sample(list(product_groups.values()), k=random.randint(1, 2)):
        items.update(random.sample(group, k=random.randint(3, min(5, len(group)))))
    for item in items:
        orders.append({"product": item, "order_id": order_id})

df = pl.DataFrame(orders)
print(f"Orders: {df['order_id'].n_unique()}, Products: {df['product'].n_unique()}")

## 2. Compute Associations & Communities

In [ ]:
assoc = combinatorial_associations(df, column_items="product", column_tid="order_id")

# Similarity for graph algorithms
sim_df = (
    assoc
    .select("lhs", "rhs", "jaccard")
    .filter(pl.col("jaccard").is_not_null() & (pl.col("jaccard") > 0))
    .rename({"jaccard": "sim"})
)

# Detect communities
comm = communities(
    sim_df,
    column_lhs="lhs",
    column_rhs="rhs",
    column_similarity="sim",
)

print(f"Communities found: {comm['community_label'].n_unique()}")
comm.sort("community_label", "item")

## 3. t-SNE Embedding

Convert Jaccard similarity to distance (`1 - similarity`), then project to 2D.

In [ ]:
# Convert similarity to distance
dist_df = (
    sim_df
    .with_columns((1.0 - pl.col("sim")).alias("distance"))
    .select("lhs", "rhs", "distance")
)

# Run t-SNE
coords = embedding(
    dist_df,
    column_lhs="lhs",
    column_rhs="rhs",
    column_distance="distance",
    perplexity=8.0,   # lower for small datasets
    n_iter=1000,
)

coords

## 4. Visualize

Plot items in 2D, colored by their community.

In [ ]:
import matplotlib.pyplot as plt

# Join coordinates with community labels
plot_data = coords.join(comm, on="item", how="left")

fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set1.colors
labels_seen = set()

for row in plot_data.iter_rows(named=True):
    c_label = row["community_label"] or "unknown"
    c_idx = int(c_label) if c_label.isdigit() else 0
    color = colors[c_idx % len(colors)]

    show_label = c_label not in labels_seen
    labels_seen.add(c_label)

    ax.scatter(
        row["x"], row["y"],
        c=[color], s=100, zorder=3,
        label=f"Community {c_label}" if show_label else None,
    )
    ax.annotate(
        row["item"], (row["x"], row["y"]),
        textcoords="offset points", xytext=(6, 6),
        fontsize=8,
    )

ax.legend(loc="best")
ax.set_title("Product Embedding (t-SNE) colored by Louvain Community")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

## 5. Adjusting t-SNE Parameters

- **perplexity**: Controls the balance between local and global structure (5-50). Lower values emphasize local neighborhoods.
- **n_iter**: Number of optimization iterations (1000-2000). More iterations = better convergence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, perp in zip(axes, [5.0, 10.0, 20.0]):
    coords_p = embedding(
        dist_df,
        column_lhs="lhs",
        column_rhs="rhs",
        column_distance="distance",
        perplexity=perp,
        n_iter=1000,
    )
    plot_p = coords_p.join(comm, on="item", how="left")

    for row in plot_p.iter_rows(named=True):
        c_label = row["community_label"] or "0"
        c_idx = int(c_label) if c_label.isdigit() else 0
        ax.scatter(row["x"], row["y"], c=[colors[c_idx % len(colors)]], s=80)
        ax.annotate(row["item"], (row["x"], row["y"]),
                    textcoords="offset points", xytext=(4, 4), fontsize=7)

    ax.set_title(f"perplexity={perp}")

plt.suptitle("Effect of Perplexity on t-SNE Embedding", fontsize=13)
plt.tight_layout()
plt.show()